In [4]:
import pandas as pd
import numpy as np


In [5]:
file_path = "output/global_military_cleaned_metadata.csv"

df = pd.read_csv(file_path)

df.head()


,country,rank,total_population_by_country,available_military_manpower,manpower_fit_for_military_service,manpower_reaching_military_age_annually,active_military_manpower,active_reserve_military_manpower,manpower_paramilitary,capital_cities_by_total_population,...,proven_coal_reserves_by_country,square_land_area,coastline_coverage,border_coverage,waterway_coverage,year,capital_city,continent,alliance,region
0,United States,1,341963408,150463900,124816644,4445524,1328000,799500,0,0,...,2.490000e+11,9833517,19924,12002,41009,2025,"Washington, D.C.",North America,NATO,North America
1,Russia,2,140820810,69002197,46189226,1267387,1320000,2000000,250000,0,...,1.620000e+11,17098242,37653,22407,102000,2025,Moscow,Europe/Asia,Non-NATO,Europe & Central Asia
2,China,3,1415043270,764123366,626864169,19810606,2035000,510000,625000,0,...,1.430000e+11,9596960,14500,22457,27700,2025,Beijing,Asia,Non-NATO,East Asia & Pacific
3,India,4,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,0,...,1.110000e+11,3287263,7000,13888,14500,2025,New Delhi,Asia,Non-NATO,South Asia
4,South Korea,5,52081799,26040900,21353538,416654,600000,3100000,120000,0,...,3.260000e+08,99720,2413,237,1600,2025,Seoul,Asia,Non-NATO,East Asia & Pacific


In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.columns


Rows: 145
Columns: 61


Index(['country', 'rank', 'total_population_by_country',
       'available_military_manpower', 'manpower_fit_for_military_service',
       'manpower_reaching_military_age_annually', 'active_military_manpower',
       'active_reserve_military_manpower', 'manpower_paramilitary',
       'capital_cities_by_total_population', 'aircraft_total',
       'aircraft_total_fighters', 'aircraft_total_attack_types',
       'aircraft_total_transports', 'aircraft_total_trainers',
       'aircraft_total_special_mission', 'aircraft_total_tanker_fleet',
       'aircraft_helicopters_total', 'aircraft_helicopters_attack',
       'armor_tanks_total', 'armor_apc_total',
       'armor_self_propelled_guns_total', 'armor_towed_artillery_total',
       'armor_mlrs_total', 'navy_ships', 'navy_force_by_tonnage',
       'navy_aircraft_carriers', 'navy_helo_carriers', 'navy_submarines',
       'navy_destroyers', 'navy_frigates', 'navy_corvettes',
       'navy_patrol_coastal_craft', 'navy_mine_warfare_craft',
       

In [7]:
df["total_personnel"] = (
    df["active_military_manpower"] +
    df["active_reserve_military_manpower"]
)


In [8]:
df["assets_per_capita"] = (
    (df["aircraft_total"] +
     df["armor_tanks_total"] +
     df["navy_ships"]) /
    df["total_population_by_country"]
)



In [9]:
df["defense_budget_to_gdp_ratio"] = (
    df["defense_spending_budget"] /
    df["purchasing_power_parity"]
)


In [10]:
df["personnel_density"] = (
    df["total_personnel"] /
    df["total_population_by_country"]
)


In [11]:
df["defense_budget_per_soldier"] = (
    df["defense_spending_budget"] /
    df["active_military_manpower"]
)


In [12]:
usa_rank = df.loc[df["country"] == "United States", "rank"].iloc[0]

df["power_index_rank_gap"] = df["rank"] - usa_rank


In [13]:
df["air_power_ratio"] = (
    df["aircraft_total"] /
    df["active_military_manpower"]
)


In [14]:
df["armor_intensity_index"] = (
    df["armor_tanks_total"] /
    df["square_land_area"]
)


In [15]:
df["naval_strength_per_coastline"] = (
    df["navy_ships"] /
    df["coastline_coverage"]
)


In [16]:
df["military_burden_index"] = (
    df["defense_spending_budget"] /
    df["total_population_by_country"]
)


In [17]:
# Total assets per country
df["total_assets"] = (
    df["aircraft_total"] +
    df["armor_tanks_total"] +
    df["navy_ships"]
)
# NOTE:
# Coalition strength is a dynamic KPI.
# This column represents coalition strength based on the
# current alliance grouping and should be recalculated
# if grouping criteria changes.

# Alliance-based coalition strength
alliance_coalition_strength = (
    df.groupby("alliance")["total_assets"]
    .sum()
    .reset_index()
    .rename(columns={"total_assets": "alliance_coalition_strength_index"})
)

# Merge back to main dataframe
df = df.merge(
    alliance_coalition_strength,
    on="alliance",
    how="left"
)


In [24]:
# NOTE:
# Coalition strength is a dynamic KPI.
# This column represents coalition strength based on the
# current alliance grouping and should be recalculated
# if grouping criteria changes.


# Manually selected coalition countries
selected_countries = [
    "United States",
    "United Kingdom",
    "France",
    "Germany",
    "India"
]

# Filter selected countries
manual_coalition_df = df[df["country"].isin(selected_countries)]

# Calculate coalition strength
manual_coalition_strength = manual_coalition_df["total_assets"].sum()

manual_coalition_strength


np.int64(28074)

In [19]:
df["manual_coalition_strength_index"] = np.where(
    df["country"].isin(selected_countries),
    manual_coalition_strength,
    np.nan
)


In [20]:
import numpy as np

df.replace([np.inf, -np.inf], np.nan, inplace=True)


In [21]:
df[
    [
        "assets_per_capita",
        "defense_budget_to_gdp_ratio",
        "personnel_density",
        "defense_budget_per_soldier",
        "power_index_rank_gap",
        "air_power_ratio",
        "armor_intensity_index",
        "naval_strength_per_coastline",
        "military_burden_index",
        "alliance_coalition_strength_index",
        "manual_coalition_strength_index"
    ]
].head()


,assets_per_capita,defense_budget_to_gdp_ratio,personnel_density,defense_budget_per_soldier,power_index_rank_gap,air_power_ratio,armor_intensity_index,naval_strength_per_coastline,military_burden_index,alliance_coalition_strength_index,manual_coalition_strength_index
0,0.000053,0.036235,0.006221,673945.783133,0,0.009822,0.000472,0.022084,2617.239094,28210,28074.0
1,0.000074,0.021649,0.023576,95454.545455,1,0.003252,0.000336,0.011128,894.754121,99736,NaN
2,0.000008,0.008558,0.001799,131203.931204,2,0.001626,0.000709,0.052000,188.686810,99736,NaN
3,0.000005,0.005725,0.001853,51526.914225,3,0.001531,0.001278,0.041857,53.224394,99736,28074.0
4,0.000078,0.017672,0.071042,77166.666667,4,0.002653,0.022423,0.094074,888.986189,99736,NaN


In [22]:
df[
    [
        "country",
        "alliance",
        "alliance_coalition_strength_index",
        "manual_coalition_strength_index"
    ]
].head(10)


,country,alliance,alliance_coalition_strength_index,manual_coalition_strength_index
0,United States,NATO,28210,28074.0
1,Russia,Non-NATO,99736,NaN
2,China,Non-NATO,99736,NaN
3,India,Non-NATO,99736,28074.0
4,South Korea,Non-NATO,99736,NaN
5,United Kingdom,NATO,28210,28074.0
6,France,NATO,28210,28074.0
7,Japan,Non-NATO,99736,NaN
8,Turkey,Non-NATO,99736,NaN
9,Italy,NATO,28210,NaN


In [23]:
output_path = "output/global_military_task3_derived_kpi_metrics.csv"

df.to_csv(output_path, index=False)

print("✅ Task 3 completed successfully")
print("📁 File saved at:", output_path)


✅ Task 3 completed successfully
📁 File saved at: output/global_military_task3_derived_kpi_metrics.csv
